# Strategy research loop — reference notebook

This is the smallest end-to-end flow a researcher runs every day:

1. Load a captured Parquet partition
2. Pick a strategy + sleeve TOML
3. Run a backtest (with optional parameter override)
4. Summarize the report
5. (Optional) compare two parameter choices side by side
6. (Optional) plot equity

All of this goes through `eventcontracts.research`. The CLI does the same things in batch (`eventcontracts backtest`, `eventcontracts sweep`); this notebook is for **interactive exploration**.

Prerequisites:
- A normalized Parquet partition produced by `eventcontracts capture` + `eventcontracts normalize` (or the in-test fixtures under `python/tests/`).
- `eventcontracts` installed editable (`pip install -e ./python`).

In [ ]:
from pathlib import Path
from eventcontracts.research import (
    backtest_one,
    compare_runs,
    load_partition_summary,
    load_sweep_results,
    summarize_report,
)

REPO_ROOT = Path.cwd().resolve().parent  # adjust if you run elsewhere

## 1. Inspect your data partition

Point `DATA_ROOT` at the output of `eventcontracts capture --normalize`. The default below assumes you captured under `./data` from the repo root.

In [ ]:
DATA_ROOT = REPO_ROOT / "data"
if DATA_ROOT.exists():
    print(load_partition_summary(DATA_ROOT))
else:
    print(f"No data at {DATA_ROOT}. Run `eventcontracts capture --normalize ...` first.")

## 2. Run one backtest

Strategy + sleeve TOMLs live under `configs/`. The example threshold strategy is the simplest entry point.

In [ ]:
STRATEGY = REPO_ROOT / "configs/strategies/example-threshold.toml"
SLEEVE = REPO_ROOT / "configs/sleeves/example-kalshi-paper.toml"

result = backtest_one(STRATEGY, SLEEVE, DATA_ROOT)
print(summarize_report(result.report))

## 3. Override a parameter without editing the TOML

Useful for quick "what if" iterations inside the notebook. The TOML on disk doesn't change.

In [ ]:
lower = backtest_one(
    STRATEGY,
    SLEEVE,
    DATA_ROOT,
    parameter_overrides={"buy_below": "0.40", "size": "3"},
)
print(compare_runs({"default": result.report, "buy_below_0.40": lower.report}))

## 4. Load a sweep results table

After running `eventcontracts sweep --out results.parquet`, this loads the rows for ranking and plotting in the notebook. Each row carries `params_json` so you can slice by parameter values.

In [ ]:
RESULTS = REPO_ROOT / "results.parquet"
if RESULTS.exists():
    rows = load_sweep_results(RESULTS)
    print(f"{len(rows)} rows; first row params: {rows[0]['params']}")
else:
    print("No sweep results yet. Try:\n"
          "  eventcontracts sweep --strategy ... --sleeve ... --params grid.toml \\\n"
          "    --windows windows.toml --data data --out results.parquet")

## 5. (Optional) plot equity

Requires matplotlib in your research environment (not a framework dep). `plot_equity` returns a matplotlib `Axes` you can further customize.

In [ ]:
try:
    from eventcontracts.research import plot_equity

    plot_equity(result.report)
except RuntimeError as exc:
    print(exc)